### 0. Environments

In [4]:
!pip install -q sentence-transformers faiss-gpu-cu12 ir_datasets

  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.4/48.4 MB 31.3 MB/s eta 0:00:0000:0100:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 866.1/866.1 kB 49.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 149.7/149.7 kB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.6/45.6 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 67.2 MB/s eta 0:00:00


In [ ]:
import ir_datasets
from tqdm import tqdm
from itertools import islice
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer
import json
import os
import torch
import gc

In [3]:
# configurations
SAVE_PATH: str = './kaggle/working/artifacts'
os.makedirs(SAVE_PATH, exist_ok=True)
EMB_PATH = os.path.join(SAVE_PATH, 'embeddings.npy')
METADATA_PATH = os.path.join(SAVE_PATH, 'doc_ids.jsonl')
FAISS_INDEX_PATH = os.path.join(SAVE_PATH, 'faiss.index')

EMBEDDING_MODEL: str = 'BAAI/bge-small-en-v1.5'
BATCH_SIZE: int = 256
DEVICE = ['cuda:0', 'cuda:1']
NORMALIZE_EMB: bool = True

# config for dry test
QUICK_RUN: bool = True
MAX_SAMPLES: int = 100

### 1. Dataset

In [6]:
!cp -r /kaggle/input/datasets/hongkimgip/msmarco-passage/msmarco-passage /root/.ir_datasets/msmarco-passage

In [4]:
def preview_iter(title, iterator, fields, n=5):
    print(f"\n========== {title} ==========")

    for i, item in enumerate(islice(iterator, n), start=1):
        print(f"\n--- {title[:-1].title()} {i} ---")
        for field in fields:
            value = getattr(item, field)
            if field == "text":
                value = value[:500]
            print(f"{field}:", value)


def download_and_preview_msmarco(
    corpus_id="msmarco-passage",
    eval_id="msmarco-passage/dev/small",
    n_samples=1,
):
    passages = ir_datasets.load(corpus_id)
    eval_set = ir_datasets.load(eval_id)

    preview_iter(
        "SAMPLE PASSAGES",
        passages.docs_iter(),
        fields=["doc_id", "text"],
        n=n_samples,
    )

    preview_iter(
        "SAMPLE QUERIES",
        eval_set.queries_iter(),
        fields=["query_id", "text"],
        n=n_samples,
    )

    preview_iter(
        "SAMPLE QRELS",
        eval_set.qrels_iter(),
        fields=["query_id", "doc_id", "relevance"],
        n=n_samples,
    )

    corpus = {}
    queries = {}
    qrels = {}

    print("\n========== LOADING FULL CORPUS ==========")
    for doc in tqdm(passages.docs_iter(), desc="Loading passages"):
        corpus[doc.doc_id] = doc.text

    print(f"Total passages loaded: {len(corpus):,}")

    print("\n========== LOADING QUERIES ==========")
    for query in tqdm(eval_set.queries_iter(), desc="Loading queries"):
        queries[query.query_id] = query.text

    print(f"Total queries loaded: {len(queries):,}")

    print("\n========== LOADING QRELS ==========")
    for qrel in tqdm(eval_set.qrels_iter(), desc="Loading qrels"):
        qrels.setdefault(qrel.query_id, {})[qrel.doc_id] = qrel.relevance

    print(f"Total qrels queries loaded: {len(qrels):,}")

    return corpus, queries, qrels


corpus, _, __ = download_and_preview_msmarco()


========== SAMPLE PASSAGES ==========


[INFO] Please confirm you agree to the MSMARCO data usage agreement found at <http://www.msmarco.org/dataset.aspx>
[INFO] If you have a local copy of https://msmarco.z22.web.core.windows.net/msmarcoranking/collectionandqueries.tar.gz, you can symlink it here to avoid downloading it again: /home/rmits/.ir_datasets/downloads/31644046b18952c1386cd4564ba2ae69


[INFO] [starting] https://msmarco.z22.web.core.windows.net/msmarcoranking/collectionandqueries.tar.gz
[INFO] [finished] https://msmarco.z22.web.core.windows.net/msmarcoranking/collectionandqueries.tar.gz: [12:45] [1.06GB] [1.38MB/s]
[INFO] [starting] fixing encoding                                                                               
[INFO] [finished] fixing encoding: [06:08] [3.06GB] [8.30MB/s]



--- Sample Passage 1 ---
doc_id: 0
text: The presence of communication amid scientific minds was equally important to the success of the Manhattan Project as scientific intellect was. The only cloud hanging over the impressive achievement of the atomic researchers and engineers is what their success truly meant; hundreds of thousands of innocent lives obliterated.

========== SAMPLE QUERIES ==========

--- Sample Querie 1 ---
query_id: 1048585
text: what is paula deen's brother

========== SAMPLE QRELS ==========

--- Sample Qrel 1 ---
query_id: 300674
doc_id: 7067032
relevance: 1

========== LOADING FULL CORPUS ==========


Loading passages: 8841823it [00:22, 398539.72it/s]


Total passages loaded: 8,841,823

========== LOADING QUERIES ==========


Loading queries: 6980it [00:00, 645960.94it/s]


Total queries loaded: 6,980

========== LOADING QRELS ==========


Loading qrels: 7437it [00:00, 605631.28it/s]

Total qrels queries loaded: 6,980


### 2. Create embeddings

In [7]:
# =========================
# Helpers
# =========================

def batched_doc_iter(corpus, batch_size, max_samples=None):
    """
    Yield batches of (doc_ids, texts) without materializing the full corpus texts.
    corpus is assumed to be dict-like: {doc_id: text}
    """
    iterator = corpus.items()

    if max_samples is not None:
        iterator = islice(iterator, max_samples)

    batch_doc_ids = []
    batch_texts = []

    for doc_id, text in iterator:
        batch_doc_ids.append(doc_id)
        batch_texts.append(text)

        if len(batch_doc_ids) == batch_size:
            yield batch_doc_ids, batch_texts
            batch_doc_ids = []
            batch_texts = []

    if batch_doc_ids:
        yield batch_doc_ids, batch_texts


# =========================
# 1. Load model
# =========================

# Không set device='cuda' ở đây.
# Multi-process pool sẽ tự copy model sang từng GPU.
model = SentenceTransformer(EMBEDDING_MODEL)

embedding_dim = model.get_sentence_embedding_dimension()
# Nếu version cũ của sentence-transformers không có get_sentence_embedding_dimension,
# có thể đổi lại thành:
# embedding_dim = model.get_embedding_dimension()

num_docs = min(len(corpus), MAX_SAMPLES) if QUICK_RUN else len(corpus)

target_devices = ["cuda:0", "cuda:1"]

print("Number of docs:", num_docs)
print("Embedding dim:", embedding_dim)
print("Target devices:", target_devices)


# =========================
# 2. Create disk-backed .npy file
# =========================
# open_memmap creates a valid .npy file, but writes chunk by chunk.
# This avoids holding the full embedding matrix in RAM.

embeddings_mmap = np.lib.format.open_memmap(
    EMB_PATH,
    mode="w+",
    dtype="float32",
    shape=(num_docs, embedding_dim),
)


# =========================
# 3. Start multi-GPU pool
# =========================

pool = model.start_multi_process_pool(
    target_devices=target_devices
)


# =========================
# 4. Encode + write incrementally
# =========================

row_id = 0

try:
    with open(METADATA_PATH, "w", encoding="utf-8") as meta_f:
        pbar = tqdm(total=num_docs, desc="Encoding corpus", unit="docs")

        for batch_doc_ids, batch_texts in batched_doc_iter(
            corpus=corpus,
            batch_size=BATCH_SIZE,
            max_samples=MAX_SAMPLES if QUICK_RUN else None,
        ):
            batch_embeddings = model.encode_multi_process(
                sentences=batch_texts,
                pool=pool,
                batch_size=BATCH_SIZE,
                normalize_embeddings=NORMALIZE_EMB,
            ).astype("float32", copy=False)

            batch_size_actual = len(batch_doc_ids)
            start = row_id
            end = row_id + batch_size_actual

            embeddings_mmap[start:end] = batch_embeddings

            for i, doc_id in enumerate(batch_doc_ids):
                meta_f.write(
                    json.dumps(
                        {
                            "row_id": start + i,
                            "doc_id": str(doc_id),
                        },
                        ensure_ascii=False,
                    )
                    + "\n"
                )

            row_id = end
            pbar.update(batch_size_actual)

            # Explicit cleanup for long-running jobs
            del batch_embeddings
            del batch_texts
            del batch_doc_ids

            gc.collect()

        pbar.close()

finally:
    # Rất quan trọng: tránh worker process bị treo sau khi xong / lỗi.
    model.stop_multi_process_pool(pool)


# Flush mmap data to disk
embeddings_mmap.flush()

assert row_id == num_docs, f"Expected {num_docs}, got {row_id}"

print("Saved embeddings:", EMB_PATH)
print("Saved metadata:", METADATA_PATH)
print("Embedding shape:", embeddings_mmap.shape)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Number of docs: 100
Embedding dim: 384
Target devices: ['cuda:0', 'cuda:1']


/tmp/ipykernel_1469292/3490224899.py:39: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  embedding_dim = model.get_sentence_embedding_dimension()
Encoding corpus: 100%|██████████| 100/100 [01:37<00:00,  1.03docs/s] 
/tmp/ipykernel_1469292/3490224899.py:91: DeprecationWarning: The `encode_multi_process` method has been deprecated, and its functionality has been integrated into `encode`. You can now call `encode` with the same parameters to achieve multi-process encoding.
  batch_embeddings = model.encode_multi_process(
Encoding corpus: 100%|██████████| 100/100 [00:00<00:00, 106.99docs/s]

Saved embeddings: ./kaggle/working/artifacts/embeddings.npy
Saved metadata: ./kaggle/working/artifacts/doc_ids.jsonl
Embedding shape: (100, 384)


#### Save as FAISS index:

In [8]:
# =========================
# Build FAISS index from saved mmap embeddings
# =========================

embeddings = np.load(EMB_PATH, mmap_mode="r")

num_docs, dim = embeddings.shape

print("Loaded embeddings:", embeddings.shape)

if NORMALIZE_EMB:
    base_index = faiss.IndexFlatIP(dim)
else:
    base_index = faiss.IndexFlatL2(dim)

index = faiss.IndexIDMap2(base_index)

row_ids = np.arange(num_docs).astype("int64")

index.add_with_ids(
    np.asarray(embeddings, dtype="float32"),
    row_ids,
)

assert index.ntotal == num_docs

faiss.write_index(index, FAISS_INDEX_PATH)

print("Saved FAISS index:", FAISS_INDEX_PATH)
print("FAISS index size:", index.ntotal)

Loaded embeddings: (100, 384)
Saved FAISS index: ./kaggle/working/artifacts/faiss.index
FAISS index size: 100
